In [7]:
# In een notebook cel:
#%pip install pettingzoo[atari] multi-agent-ale-py autorom[accept-rom-license]
#%pip install tensorflow tensorflow-probability numpy matplotlib scikit-learn pandas

# Download Atari ROMs (alleen de eerste keer nodig)
!AutoROM -y

AutoROM will download the Atari 2600 ROMs.
They will be installed to:
	/usr/local/lib/python3.12/dist-packages/AutoROM/roms
	/usr/local/lib/python3.12/dist-packages/multi_agent_ale_py/roms

Existing ROMs will be overwritten.
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/multi_agent_ale_py/roms/adventure.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/multi_agent_ale_py/roms/air_raid.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/multi_agent_ale_py/roms/alien.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/multi_agent_ale_py/roms/amidar.bin
Installed /usr/local/lib/python3.12/dist-packages/AutoROM/roms/assault.bin
Installed /usr/local/lib/python3.12/dist-packages/multi_

In [8]:
# Imports
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp

def build_actor_network(state_shape, action_dim, hidden_layers=[256, 128]):
    inputs = tf.keras.Input(shape=state_shape)
    x = inputs

    for units in hidden_layers:
        x = tf.keras.layers.Dense(units, activation="relu")(x)
        x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(action_dim, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)

def build_critic_network(state_shape, hidden_layers=[256, 128]):
    inputs = tf.keras.Input(shape=state_shape)
    x = inputs

    for units in hidden_layers:
        x = tf.keras.layers.Dense(units, activation="relu")(x)
        x = tf.keras.layers.Dropout(0.2)(x)

    outputs = tf.keras.layers.Dense(1, activation="linear")(x)
    return tf.keras.Model(inputs, outputs)

# A2C Agent Implementation
class A2CAgent:
    def __init__(self, state_dim, action_dim, player_id,
             learning_rate=0.001, gamma=0.99, entropy_coef=0.01,
             value_coef=0.5, max_grad_norm=0.5, hidden_layers=[256, 128]):
        self.player_id = player_id
        self.gamma = gamma
        self.entropy_coef = entropy_coef
        self.value_coef = value_coef
        self.max_grad_norm = max_grad_norm

        self.actor = build_actor_network(state_dim, action_dim, hidden_layers)
        self.critic = build_critic_network(state_dim, hidden_layers)
        self.actor_optimizer = tf.keras.optimizers.Adam(learning_rate)
        self.critic_optimizer = tf.keras.optimizers.Adam(learning_rate)

        # Training metrics
        self.episode_rewards = []
        self.episode_losses = []

    def select_action(self, state):
        # Policy-based actie selectie
        state = np.expand_dims(state, axis=0) # Belangrijk voor Atari
        probs = self.actor(state)
        dist = tfp.distributions.Categorical(probs=probs[0])
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return int(action.numpy()[0]), log_prob

    def update(self, states, actions, rewards, dones, next_states):
        states = np.array(states)
        actions = np.array(actions)

        # Compute returns
        returns = []
        R = 0
        for r, d in zip(reversed(rewards), reversed(dones)):
            if d:
                R = 0
            R = r + self.gamma * R
            returns.insert(0, R)
        returns = tf.convert_to_tensor(returns, dtype=tf.float32)

        # Compute values and advantages
        values = tf.squeeze(self.critic(states))
        values = tf.cast(values, tf.float32)
        advantages = returns - values

        # Update actor
        with tf.GradientTape() as tape:
            probs = self.actor(states)
            dist = tfp.distributions.Categorical(probs=probs)
            actions_tensor = tf.convert_to_tensor(actions, dtype=tf.int32)
            log_probs = dist.log_prob(actions_tensor)

            actor_loss = -tf.reduce_mean(log_probs * tf.stop_gradient(advantages))
            entropy = tf.reduce_mean(dist.entropy())
            total_actor_loss = actor_loss - self.entropy_coef * entropy

        actor_grads = tape.gradient(total_actor_loss, self.actor.trainable_variables)
        actor_grads, _ = tf.clip_by_global_norm(actor_grads, self.max_grad_norm)
        self.actor_optimizer.apply_gradients(zip(actor_grads, self.actor.trainable_variables))

        # Update critic
        with tf.GradientTape() as tape:
            values_pred = tf.squeeze(self.critic(states))
            critic_loss = tf.reduce_mean(tf.square(returns - values_pred))

        critic_grads = tape.gradient(critic_loss, self.critic.trainable_variables)
        critic_grads, _ = tf.clip_by_global_norm(critic_grads, self.max_grad_norm)
        self.critic_optimizer.apply_gradients(zip(critic_grads, self.critic.trainable_variables))

        return total_actor_loss.numpy(), critic_loss.numpy()

    def train_episode(self, env, agent_name):
        states, actions, rewards, dones = [], [], [], []
        total_reward = 0

        obs, _ = env.reset()
        done = False

        while not done:
            action, _ = self.select_action(obs)
            next_obs, reward, terminated, truncated, info = env.last()
            done = terminated or truncated

            states.append(obs)
            actions.append(action)
            rewards.append(reward)
            dones.append(done)

            total_reward += reward
            obs = next_obs

        # Update networks
        if len(states) > 0:
            actor_loss, critic_loss = self.update(states, actions, rewards, dones, states)
            self.episode_rewards.append(total_reward)
            self.episode_losses.append((actor_loss, critic_loss))

        return total_reward

    def evaluate(self, env, n_episodes=10):
        eval_rewards = []

        for _ in range(n_episodes):
            obs, _ = env.reset()
            done = False
            episode_reward = 0

            while not done:
                action, _ = self.select_action(obs)
                next_obs, reward, terminated, truncated, info = env.last()
                done = terminated or truncated
                episode_reward += reward
                obs = next_obs

            eval_rewards.append(episode_reward)

        return np.mean(eval_rewards), np.std(eval_rewards)

In [14]:
# Importeer libraries
from pettingzoo.atari import warlords_v3 # Importeer warlords_v3 direct
import numpy as np
import matplotlib.pyplot as plt

# Maak environment aan (RAM mode)
env = warlords_v3.parallel_env(obs_type="ram") # Gebruik parallel_env voor meerdere agents

# Reset environment om shapes te bepalen
# Voor parallelle omgevingen retourneert reset dictionaries van observaties en info
observations, infos = env.reset()
# Haal de observatie voor de eerste agent op (aangenomen dat de A2C agent voor één speler is)
obs = observations[env.agents[0]]

state_dim = obs.shape  # (128,) voor RAM
action_dim = env.action_space(env.agents[0]).n  # Aantal acties (meestal 18)

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Agents: {env.agents}")

State dimension: (128,)
Action dimension: 6
Agents: ['first_0', 'second_0', 'third_0', 'fourth_0']


In [17]:
# Maak A2C agent aan met hyperparameters
a2c_agent = A2CAgent(
    state_dim=state_dim,
    action_dim=int(action_dim),  # Expliciet casten naar int
    player_id=0,  # Of de juiste player_id voor jouw agent
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    value_coef=0.5,
    max_grad_norm=0.5,
    hidden_layers=[256, 128]
)

print("A2C Agent geïnitialiseerd!")

A2C Agent geïnitialiseerd!


In [18]:
# Training parameters
num_episodes = 500
eval_interval = 50  # Evalueer elke X episodes
save_interval = 100  # Sla model op elke X episodes

# Training loop
print(f"Start training voor {num_episodes} episodes...")

for episode in range(num_episodes):
    # Train één episode
    episode_reward = a2c_agent.train_episode(env, "A2C")

    # Print voortgang
    if (episode + 1) % 10 == 0:
        avg_reward = np.mean(a2c_agent.episode_rewards[-10:])
        print(f"Episode {episode+1}/{num_episodes} | "
              f"Reward: {episode_reward:.2f} | "
              f"Avg (last 10): {avg_reward:.2f} | "
              f"Total Episodes: {len(a2c_agent.episode_rewards)}")

    # Evalueer periodiek
    if (episode + 1) % eval_interval == 0:
        mean_reward, std_reward = a2c_agent.evaluate(env, n_episodes=10)
        print(f"  --- Evaluation: {mean_reward:.2f} ± {std_reward:.2f} ---")

    # Sla model op periodiek
    if (episode + 1) % save_interval == 0:
        a2c_agent.actor.save_weights(f'a2c_actor_ep{episode+1}.h5')
        a2c_agent.critic.save_weights(f'a2c_critic_ep{episode+1}.h5')
        print(f"  --- Model saved at episode {episode+1} ---")

print("Training voltooid!")

Start training voor 500 episodes...


TypeError: Exception encountered when calling Functional.call().

[1mfloat() argument must be a string or a real number, not 'dict'[0m

Arguments received by Functional.call():
  • inputs=array([{'first_0': array([  0,   0,  33, 109,  79,  36, 107,   5,  66,  60,  60,   0,  31,
               31, 239, 240, 240, 240, 240, 240, 240,   0,   0, 255, 255, 255,
              255, 255, 255,  31,  31, 255, 255, 255, 255, 255, 255,  31,  31,
              240, 240, 240, 240, 240, 240,   0,   0, 240, 240, 240, 240, 240,
              240,   0,   0, 255, 255, 255, 255, 255, 255,  31,  31, 255, 255,
              255, 255, 255, 255,  31,  31, 240, 240, 240, 240, 240, 240,   0,
                0,   0,   0,  71,  71,  69,  69,  67,  67,  65,  65,   0,  73,
               45,  45,  45,  45, 237,   5,   5,   5,   5, 255,  11,  23,   0,
                0,   0, 195,   0, 212,  98,   0,   1,   7,   7,  20,  20,   8,
                7,   0,   0, 255, 247,   0,   0, 161,  12,  15, 248], dtype=uint8), 'se

In [ ]:
# Finale evaluatie
final_mean, final_std = a2c_agent.evaluate(env, n_episodes=20)
print(f"\nFinale Evaluatie:")
print(f"Mean Reward: {final_mean:.2f} ± {final_std:.2f}")
print(f"Total training episodes: {len(a2c_agent.episode_rewards)}")

In [ ]:
def plot_training_curves(agent):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Rewards
    rewards = agent.episode_rewards
    axes[0, 0].plot(rewards, alpha=0.3, label='Raw Rewards')

    # Moving average
    window = 20
    if len(rewards) >= window:
        moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
        axes[0, 0].plot(range(window-1, len(rewards)), moving_avg,
                        label=f'Moving Avg (window={window})', linewidth=2)

    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Reward')
    axes[0, 0].set_title('Training Rewards')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Losses
    if agent.episode_losses:
        actor_losses = [loss[0] for loss in agent.episode_losses]
        critic_losses = [loss[1] for loss in agent.episode_losses]

        axes[0, 1].plot(actor_losses, label='Actor Loss', alpha=0.7)
        axes[0, 1].plot(critic_losses, label='Critic Loss', alpha=0.7)
        axes[0, 1].set_xlabel('Episode')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].set_title('Training Losses')
        axes[0, 1].legend()
        axes[0, 1].grid(True)

    # Reward distribution
    axes[1, 0].hist(rewards, bins=30, alpha=0.7, edgecolor='black')
    axes[1, 0].set_xlabel('Reward')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title('Reward Distribution')
    axes[1, 0].grid(True)

    # Training stability (rolling std)
    if len(rewards) >= window:
        rolling_std = []
        for i in range(window, len(rewards)+1):
            rolling_std.append(np.std(rewards[i-window:i]))
        axes[1, 1].plot(range(window, len(rewards)+1), rolling_std)
        axes[1, 1].set_xlabel('Episode')
        axes[1, 1].set_ylabel('Rolling Std Dev')
        axes[1, 1].set_title(f'Training Stability (window={window})')
        axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()

# Plot de resultaten
plot_training_curves(a2c_agent)

In [ ]:
# Sla het finale model op
a2c_agent.actor.save_weights('a2c_actor_final.h5')
a2c_agent.critic.save_weights('a2c_critic_final.h5')

# Sla ook de hyperparameters op
import json
hyperparams = {
    'learning_rate': 0.001,
    'gamma': 0.99,
    'entropy_coef': 0.01,
    'value_coef': 0.5,
    'max_grad_norm': 0.5,
    'hidden_layers': [256, 128],
    'num_episodes': len(a2c_agent.episode_rewards),
    'final_mean_reward': final_mean,
    'final_std_reward': final_std
}

with open('a2c_hyperparams.json', 'w') as f:
    json.dump(hyperparams, f, indent=4)

print("Model en hyperparameters opgeslagen!")

In [ ]:
# Laad een getraind model
loaded_agent = A2CAgent(
    state_dim=state_dim,
    action_dim=action_dim,
    player_id=0,
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    hidden_layers=[256, 128]
)

# Laad de weights
loaded_agent.actor.load_weights('a2c_actor_final.h5')
loaded_agent.critic.load_weights('a2c_critic_final.h5')

print("Model geladen!")

In [ ]:
# In het tournament notebook, vervang de PPO agent import:
from A2C_agent import A2CAgent

# Laad het getrainde model
a2c_trained = A2CAgent(
    state_dim=(128,),
    action_dim=18,
    player_id=0,
    learning_rate=0.001,
    gamma=0.99,
    entropy_coef=0.01,
    hidden_layers=[256, 128]
)

a2c_trained.actor.load_weights('a2c_actor_final.h5')
a2c_trained.critic.load_weights('a2c_critic_final.h5')

# Voeg toe aan agent_instances
agent_instances = [
    RandomAgent(),
    HeuristicAgent(),
    a2c_trained,  # Je getrainde A2C agent
    RandomAgent()
]

agent_names = ['RandomAgent', 'HeuristicAgent', 'A2CAgent', 'Agent4']